# មេរៀន ១៣ - អង្គចងចាំភ្នាក់ងារជាមួយក្រាហ្វព័ត៌មាន Cognee Knowledge


## ការតំឡើង

សៀវភៅប្រតិបត្តិបង្ហាញពីវិធីសាស្ត្រកសាង **ជំនួយការកូដ** ឆ្លាតវៃមួយដែលមានចងចាំប្រកបដោយភាពច្បាស់លាស់ជាមួយនឹងបណ្ដាញចំណេះដឹង [**Cognee**](https://www.cognee.ai/) និង **Microsoft Agent Framework** (MAF)។

Cognee បម្លែងអត្ថបទដែលមិនមានរចនាសម្ព័ន្ធទៅជាបណ្ដាញចំណេះដឹងមានរចនាសម្ព័ន្ធដែលអាចសួរបានដោយធានាដោយការបញ្ចូលវ៉ិចទ័រ — ផ្តល់ឲ្យភ្នាក់ងាររបស់អ្នកមានចងចាំមួយដែលស្គាល់ទំនាក់ទំនងយ៉ាងសំបូរបែប។

### អ្វីខ្លះដែលអ្នកនឹងរៀន
1. **សាងសង់បណ្ដាញចំណេះដឹង**៖ បម្លែងប្រវត្តិរូបអ្នកអភិវឌ្ឍន៍ និងគោលការណ៍ល្អៗ ទៅជាចំណេះដឹងដែលមានរចនាសម្ព័ន្ធ និងអាចសួរបាន។
2. **រួមបញ្ចូល Cognee ជាមួយ MAF**៖ ប្រើមុខងារ `@tool` ដើម្បីអនុញ្ញាតឲ្យភ្នាក់ងារ MAF សួរបណ្ដាញចំណេះដឹងរបស់ Cognee។
3. **ការសន្ទនាមានការយល់ដឹងពីវគ្គសម័យ**៖ រក្សាមContext តាមរយៈសំណួរជាច្រើនក្នុងវគ្គសម័យដូចគ្នា។
4. **ចងចាំរយៈពេលវែង**៖ រក្សាទុកចំណេះដឹងសំខាន់ៗរយៈពេលវែង និងទាញយកវាជាសន្ទនាថ្មី។

### របស់ត្រូវមានរួចហើយ
- Python 3.9+
- Redis កំពុងរត់នៅក្នុងកុំព្យូទ័រខ្លួនឯង (`docker run -d -p 6379:6379 redis`) សម្រាប់គ្រប់គ្រងវគ្គសម័យ
- គន្លឹះ API LLM មួយ (ឧ. OpenAI) — កំណត់ `LLM_API_KEY` ក្នុង `.env`
- `CACHING=true` នៅក្នុង `.env` (តម្រូវសម្រាប់វគ្គសម័យ Cognee)
- គម្រោង Microsoft Foundry ជាមួយម៉ូដែលជជែកដែលបានដាក់ចេញ
- `AZURE_AI_PROJECT_ENDPOINT` និង `AZURE_AI_MODEL_DEPLOYMENT_NAME` នៅក្នុង `.env`
- បាន authenticate អ្នកប្រើ Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## ប្រភេទនៃចងចាំភ្នាក់ងារ

សៀវភៅកំណត់ត្រានេះសិក្សាគ្រប់ប្រភេទចងចាំទាំងបីដូចក្នុងសៀវភៅកំណត់ត្រាបង្រៀន Lesson 13 នៃមេរៀន​ធំ តែប្រើCognee ជារបារទិន្នន័យចងចាំរយៈពេលវែង៖

| ប្រភេទចងចាំ | ប្រព័ន្ធ | អាយុកាល |
|---|---|---|
| **កំពុងដំណើរការ** | `agent.create_session()` (MAF) | ការសន្ទនាតែមួយ |
| **រយៈពេលខ្លី** | តំបន់សម័យសូម្បី Cognee (Redis) | សម័យសូម្បីតែមួយ |
| **រយៈពេលវែង** | ក្រាហ្វ knowledge Cognee + vectors | ទីផ្សារប្រចាំ |

### ស្ថាបត្យកម្មចងចាំរបស់ Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## រៀបចំផ្ទុក Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## ផ្នែក 1 — កសាងមូលដ្ឋានចំណេះដឹង

យើងចំណាយពេលបញ្ចូលទិន្នន័យបីប្រភេទដើម្បីបង្កើតមូលដ្ឋានចំណេះដឹងទូលំទូលាយសម្រាប់ជំនួយកូដរបស់យើង៖

1. **ប្រវត្តិម្ចាស់កម្មវិធី** — ជំនាញផ្ទាល់ខ្លួន និង​ បរិភោគបច្ចេកទេស
2. **ការអនុវត្តល្អបំផុតក្នុង Python** — Zen របស់ Python ជាមួយនឹងមគ្គុទេសក៍អនុវត្ត
3. **ការសន្ទនាថ្ងៃពីមុន** — សម័យសំណួរនិងចម្លើយចំរូងរវាងអ្នកអភិវឌ្ឍន៍ និងជំនួយការប្រើប្រាស់ AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## បង្ហាញទិដ្ឋភាព Knowledge Graph

Cognee អាចបង្ហាញការមើលឃើញ HTML អន្តរកម្មនៃអង្គភាព និងទំនាក់ទំនងដែលវាបានទាញយក។ 


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## ពង្រីកការចងចាំជាមួយ Memify

`memify()` វិភាគក្រាបចំណេះដឹងហើយបង្កើតក្រមសេចក្ដីណែនាំឆ្លាតវៃ — កំណត់រចនាសម្ព័ន្ធ ការអនុវត្តល្អបំផុត និងទំនាក់ទំនងរវាងយោបល់។


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ផ្នែក 2 — អេហ្សង់ MAF ជាមួយឧបករណ៍ Cognee

ឥឡូវនេះយើងបង្កើតអេហ្សង់ MAF ដែលអាចសំនួរទៅកាន់​អាក្រក់ចំណេះដឹង​របស់ Cognee តាមរយៈមុខងារ `@tool` ។ វាអនុញ្ញាតិឲ្យអេហ្សង់ប្រើអំណាចពេញលេញនៃការស្វែងរកមានន័យតាមក្រាហ្វបែប graph-aware ខណៈដែលរក្សាទុកបរិបទការពិភាក្សាតាមរយៈសម័យ។


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## អនុស្សារណៈធ្វើការ ជាមួយសម័យ

`AgentSession` (ដែលបានបង្កើតតាមរយៈ `agent.create_session()`) ផ្តល់អនុស្សារណៈធ្វើការក្នុងសម័យមួយ។ អ្នកតំណាងអាចយោងត្រឡប់ទៅសារមុនៗ ខណៈដែលក៏កំពុងសួរព័ត៌មានពីក្រាហ្វ knowledge រយៈពេលវែងរបស់ Cognee ផងដែរ។


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## សម័យថ្មី — អនុក្ខរកម្មរយៈបច្ចុកาลនៅតែមាន

ការចាប់ផ្តើមសម័យថ្មីនៅជំហានដំណើរការធ្វើអោយភាពចងចាំការងារបានជ្រាបស្រួល ប៉ុន្តែក្រាបចំណេះដឹង Cognee នៅតែអាចប្រើបាន។ តំណាងអាចយកចំណេះដឹងរយៈបច្ចុកាលដដែលក្នុងការប្រសិទ្ធិការថ្មីមួយបាន។


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## សេចក្តីសង្ខេប

ក្នុងសៀវភៅកំណត់ត្រានេះ អ្នកបានសាងសង់ជំនួយការកូដដែលបញ្ចូលចំណេះដឹង **អង្គចងចាំបំភ្លឺការងារ MAF** (`agent.create_session()`) ជាមួយ **ក្រាបចំណេះដឹងរយៈពេលវែង Cognee**។

### អ្វីដែលអ្នកបានរៀន
1. **សំណុំបង្កើតក្រាបចំណេះដឹង**៖ Cognee ស្រូបយកអត្ថបទមិនរចនាសម្ព័ន្ធ និងបង្កើតក្រាប + អង្គចងចាំវ៉ិចទ័រ។
2. **ការបន្ថែមតម្លៃក្រាបជាមួយ memify**៖ ទ្រឹស្តីដែលបានទាញយក និងទំនាក់ទំនងសម្បូរបែបលើក្រាបដែលមានរួចរបស់អ្នក។
3. **ការរួមបញ្ចូល MAF + Cognee**៖ មុខងារ `@tool` អនុញ្ញាតឲ្យភ្នាក់ងារក្នុង MAF សួរច្បាប់ក្រាប Cognee ជារបៀបទាមទារ។
4. **អង្គចងចាំបំភ្លឺការងារ + អង្គចងចាំរយៈពេលវែង**៖ `AgentSession` (តាមរយៈ `agent.create_session()`) ផ្តល់បរិបទសម័យខណៈដែល Cognee ផ្តល់ចំណេះដឹងឲ្យមានភាពជាប់ទាក់ទង។
5. **ការស្វែងរកតាមការត្រងជាមួយ NodeSets**៖ គោលដៅទៅកាន់សំណុំផ្នែកខ្លះនៃក្រាបចំណេះដឹង (ឧ. គោលការណ៍តែប៉ុណ្ណោះ)។

### ចំណុចសំខាន់ដែលត្រូវយល់
- **Cognee** បម្លែងអត្ថបទស្រឡាយទៅជាអង្គចងចាំដែលមានរចនាសម្ព័ន្ធ និងយល់ពីទំនាក់ទំនង — ខ្លាំងជាងហាងវ៉ិចទ័រលាយ។
- **មុខងារ `@tool`** ជាស្ពានប្រព័ន្ធភ្នាក់ងារមាផនិងប្រព័ន្ធចំណេះដឹងខាងក្រៅយ៉ាងល្អ។
- **`AgentSession`** (តាមរយៈ `agent.create_session()`) រក្សាបរិបទពិភាក្សាផ្ទាល់ខ្លួនបំបែកពីចំណេះដឹងរយៈពេលវែង។
- ក្រាបចំណេះដឹងដដែលប្រើបានសម្រាប់សម័យនិងភ្នាក់ងារច្រើន។

### កម្មវិធីពិតប្រាកដក្នុងជីវិត
- **ជំនួយការសម្របសម្រួលអ្នកអភិវឌ្ឍន៍**៖ ពិនិត្យកូដ, វិភាគគ្រោះហត្ថការ, ជំនួយការអាគារស្ថាបត្យកម្ម
- **ជំនួយការប្រកួតប្រជែងជាមួយអតិថិជន**៖ ជំនួយភ្នាក់ងារលើឯកសារផលិតផល, សំណួរដែលសួរញឹកញាប់ និងកំណត់ត្រា CRM
- **ជំនួយការជំនាញខាងក្នុង**៖ ជំនួយបញ្ហារដ្ឋប្បវេណ, ច្បាប់ ឬសន្តិសុខដោយយល់ព្រោះលើមគ្គុទេសក៍
- **ស្រទាប់ទិន្នន័យរួម**៖ បញ្ចូលទិន្នន័យរចនាសម្ព័ន្ធនិងមិនរចនាសម្ព័ន្ធទៅក្នុងក្រាបដែលអាចស៊ើបអង្កេតបានមួយ

### ជំហានបន្ទាប់
- សាកល្បងវិញ្ញាណបន្ថែមពេលវេលាក្នុង Cognee
- កំណត់អង់តូឡូហ្ស៊ី OWL សម្រាប់គុណភាពក្រាបផ្នែកដែនជាក់លាក់
- បន្ថែមចរន្តមតិអ្នកប្រើដើម្បីធ្វើឲ្យការយកសំណត្ររយៈពេលប្រសើរឡើង
- បង្រួមទៅប្រព័ន្ធភ្នាក់ងារច្រើនបំបែកស្រទាប់អង្គចងចាំ Cognee ដែរតែមួយ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
